In [1]:
import numpy as np
import pandas as pd
from datetime import date
import yaml
from pathlib import Path
import os

In [2]:
def mpr(state, yaml_path, local_rc1_path):
    #load yaml files first
    with open((yaml_path / "d_ro_distt.yaml"), 'r') as f:
        d_ro_distt = yaml.safe_load(f)
    with open((yaml_path / "d_ro_nm.yaml"), 'r') as f:
        d_ro_nm = yaml.safe_load(f)
    with open((yaml_path / "d_ST.yaml"), 'r') as f:
        d_ST = yaml.safe_load(f)
    with open((yaml_path / "d_state_distt_nm.yaml"), 'r') as f:
        d_state_distt_nm = yaml.safe_load(f)
    with open((yaml_path / "d_state_gsheet.yaml"), 'r') as f:
        d_state_gsheet = yaml.safe_load(f)
    with open((yaml_path / "d_state_nm.yaml"), 'r') as f:
        d_state_nm = yaml.safe_load(f)
    with open((yaml_path / "d_state_ro.yaml"), 'r') as f:
        d_state_ro = yaml.safe_load(f)
    with open((yaml_path / "d_state_season_nm.yaml"), 'r') as f:
        d_state_season_nm = yaml.safe_load(f)
    
    today = date.today()
    print(f"\n{d_state_nm[state]} : MPR as on {today}")
    print(f'{" "*30}: RECEIVED')
    df = pd.read_excel(local_rc1_path, sheet_name="RC1")
    df1 = df[(df['STAT'] == state)]

    for season,season_nm in d_state_season_nm[state].items():
        print(f"  {season_nm}")
        for ST,ST_nm in d_ST.items():
            if ST == 1:
                central_total = df1[(df1['ST']==ST) & (df1['SESON']==season)].shape[0]
                central_received = 0
                for ro in d_state_ro[state]:
                    central_received += mpr_ro(df1, season, ST, ro, d_ro_distt, d_ro_nm)
                print((f"   Central Total".ljust(30))+(f": {central_total}"))
            elif ST == 2:
                state_total = df1[(df1['ST']==ST) & (df1['SESON']==season)].shape[0]
                state_received = mpr_state(df1, season, ST)
                print((f"   State Total".ljust(30))+(f": {state_total}\n"))

In [3]:
def mpr_ro(df, season, ST, ro, d_ro_distt, d_ro_nm):
    dist_list = d_ro_distt[ro]
    ro_total = df[(df['SESON'] == season)
                & (df['ST'] == ST)
                & (df['DIST'].isin(dist_list))]
    ro_received = ro_total.shape[0]
    print((f"      RO {d_ro_nm[ro]}".ljust(30))+(f": {ro_received}"))
    return ro_received

In [4]:
def mpr_state(df, season, ST):
    state_total = df[(df['SESON'] == season)
                & (df['ST'] == ST)]
    state_received = state_total.shape[0]
    return state_received

In [5]:
def main(state):
    yaml_path = Path.cwd() / "09_yaml_data"
    local_rc1_path = Path.cwd() / "08_Generated_RCs" / "RC1.xlsx"
    for state in states:
        mpr(state, yaml_path, local_rc1_path)

In [6]:
states = [34]

In [7]:
main(states)


Puducherry : MPR as on 2026-08-03
                              : RECEIVED
  Kharif
      RO Chennai              : 19
   Central Total              : 19
   State Total                : 0

  Rabi-I
      RO Chennai              : 0
   Central Total              : 0
   State Total                : 0

  Rabi-II
      RO Chennai              : 0
   Central Total              : 0
   State Total                : 0

